Creare un'applicazione in Python che utilizza MySQL per gestire un sistema di gestione per una libreria, includendo la gestione del prestito dei libri.

Setup del progetto:

Installare le librerie necessarie (mysql.connector per interagire con Mysql).
Creare un database chiamato libreria e una tabella chiamata libri opz(utenti, prestiti).

Definizione del libro: Ogni libro deve avere i seguenti attributi:

titolo (string)
autore (string)
anno_pubblicazione (int)
genere (string)
disponibile (boolean)
prestiti (lista di oggetti che contengono informazioni sul prestito, come nome del prestatario e data del prestito)

Funzionalità dell'applicazione:

Aggiungi libro: Implementare una funzione che consente di aggiungere un nuovo libro alla collezione.
Visualizza libri: Implementare una funzione che mostra tutti i libri presenti nella collezione, evidenziando se sono disponibili o meno.
Modifica libro: Implementare una funzione che consente di modificare le informazioni di un libro esistente (identificato da titolo o autore).
Elimina libro: Implementare una funzione per rimuovere un libro dalla collezione.
Prestito libro: Implementare una funzione che consente di prestare un libro a un utente. Dovrà aggiornare lo stato del libro e registrare le informazioni del prestito.
Restituzione libro: Implementare una funzione che consente di restituire un libro, aggiornando lo stato del libro e rimuovendo il prestito dalla lista.

Interfaccia utente:

Creare un semplice menu a console che permetta all'utente di scegliere quale operazione eseguire (aggiungere, visualizzare, modificare, eliminare, prestare o restituire un libro).
Gestire le eccezioni per garantire che l'input dell'utente sia valido.


CREATE DATABASE IF NOT EXISTS libreria;

CREATE TABLE IF NOT EXISTS libreria.books (
	id INT NOT NULL AUTO_INCREMENT PRIMARY KEY,
    titolo VARCHAR(100) NOT NULL,
    autore VARCHAR(50) NOT NULL,
    anno_pubblicazione INT NOT NULL,
    genere VARCHAR(50) NULL,
    disponibile BOOLEAN DEFAULT TRUE
);

CREATE TABLE IF NOT EXISTS libreria.loans (
	id INT NOT NULL AUTO_INCREMENT PRIMARY KEY,
    utente VARCHAR(50) NOT NULL,
    libro INT NOT NULL,
    data_prestito TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    data_restituzione DATETIME NULL,
    CONSTRAINT FK_loans_books 
		FOREIGN KEY(libro) REFERENCES libreria.books(id)
        ON DELETE CASCADE
        ON UPDATE CASCADE
);

INSERT INTO libreria.books (titolo, autore, anno_pubblicazione, genere) 
	VALUES ("Il tradimento", "Elle Kennedy", 2024, "COMMEDIA"),
		   ("Game of Olympus", "Hazel Riley", 2026, "HORROR"),
           ("Alberi", "Aya Koda", 2016, "AZIONE");
    
INSERT INTO libreria.loans (utente, libro) VALUES ("Giuseppe Verdi", 1);

UPDATE libreria.books SET titolo='Il tradimento', autore='Elle Kennedy', 
	anno_pubblicazione=2020, genere='COMMEDIA' WHERE id = 5;

SELECT * FROM libreria.books;
SELECT * FROM libreria.loans;

DELETE FROM libreria.loans WHERE id = 4;
DELETE FROM libreria.books WHERE id = 1;

UPDATE libreria.loans SET data_restituzione = '2026-05-22'
                    WHERE libro = 1 AND data_restituzione IS NULL;
                    
-- SELECT [DISTINCT] column_name1, column_name2, ... column_nameN | * | aggregate_function(expression)
-- 	FROM table_name
-- 	[WHERE search_condition]
-- 	[GROUP BY]
-- 	[HAVING search_condition]
-- 	[ORDER BY]
-- 	[LIMIT n]

SELECT district, COUNT(*) AS numCity 
	FROM sakila.address as a INNER JOIN sakila.city as c ON a.city_id = c.city_id
	WHERE a.city_id > 300
    GROUP BY district
    HAVING numCity >= 2
    ORDER BY numCity DESC
    LIMIT 5;








In [ ]:
import mysql.connector as mc
import datetime as date

# Connect to server
db = mc.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="root",
    database="libreria")

# Get a cursor
cursor = db.cursor()

def crea_tabella_books():
    sql = """CREATE TABLE IF NOT EXISTS libreria.books (
        id INT NOT NULL AUTO_INCREMENT PRIMARY KEY,
        titolo VARCHAR(100) NOT NULL,
        autore VARCHAR(50) NOT NULL,
        anno_pubblicazione INT NOT NULL,
        genere VARCHAR(50) NULL,
        disponibile BOOLEAN DEFAULT TRUE
    );"""
    cursor.execute(sql)
    print('Tabella Books creata con successo!')
    
def crea_tabella_loans():
    sql = """CREATE TABLE IF NOT EXISTS libreria.loans (
        id INT NOT NULL AUTO_INCREMENT PRIMARY KEY,
        utente VARCHAR(50) NOT NULL,
        libro INT NOT NULL,
        data_prestito TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        data_restituzione DATETIME NULL,
        CONSTRAINT FK_loans_books 
            FOREIGN KEY(libro) REFERENCES libreria.books(id)
            ON DELETE CASCADE
            ON UPDATE CASCADE
    );"""
    cursor.execute(sql)
    print('Tabella Loans creata con successo!')
    
crea_tabella_books()
crea_tabella_loans()

def add_book():
    # Aggiungi libro: Implementare una funzione che consente di 
    # aggiungere un nuovo libro alla collezione
    cursor = db.cursor()
    try:
        titolo = input('Inserisci il titole del libro: ')
        autore = input('Inserisci Autore del libro: ')
        anno = int(input('Inserisci anno di pubblicazione: '))
        genere = input('Inserisci genere( FANTASY | HORROR | THRILLER | COMMEDIA | AZIONE): ')
        if genere not in ['FANTASY', 'HORROR', 'THRILLER', 'COMMEDIA', 'AZIONE']:
            raise TypeError('Genere selezionato non valido')
        
        sql = """ INSERT INTO libreria.books (titolo, autore, anno_pubblicazione, genere) 
	              VALUES (%s, %s, %s, %s);
              """
        values = (titolo, autore, anno, genere)
        cursor.execute(sql, values) #Eseguo la query SQL passandogli i valori da inserire
        db.commit() # Completa l'operazione
        
        if cursor._last_insert_id:
            print(f'Libro {titolo} di {autore} salvato nel DB')
    except ValueError as e:
        print('Anno in formato non valido.')
        return
    except TypeError as e:
        print(e)
        
    cursor.close()

def find_books():
    # Implementare una funzione che mostra tutti i libri presenti nella collezione, 
    # evidenziando se sono disponibili o meno.
    cursor = db.cursor()
    
    sql = "SELECT * FROM libreria.books"
    cursor.execute(sql)
    
    books = cursor.fetchall()
    if not books:
        print("Nessun libro presente nella libreria")
    else:
        for book in books:
            print(f'----- {book[1]} -----')
            print(f'Autore: {book[2]}')
            print(f'Anno: {book[3]}')
            print(f'Genere: {book[4]}')
            print(f'----- {"Disponibile" if book[5] else "Non Disponibile"} -----')
            print()
            
    cursor.close()

def find_book(titolo, autore):
    # Leggere e restituire un libro identificato da titolo o autore
    cursor = db.cursor()
    
    sql = "SELECT * FROM libreria.books WHERE titolo = %s and autore = %s"
    cursor.execute(sql, (titolo,autore))
    
    book = cursor.fetchone()
    return book

def find_and_update_book():
    cursor = db.cursor()
    try:
        print('Quale libro vuoi modificare?')
        srcTitolo = input('Inserisci Titolo: ')
        srcAutore = input('Inserisci Autore: ')
        
        book = find_book(srcTitolo, srcAutore)
        if(book):
            titolo = input(f'Inserisci nuovo Titolo ({book[1]}): ') or book[1]
            autore = input(f'Inserisci nuovo Autore ({book[2]}): ') or book[2]
            anno_pubblicazione = int(input(f'Inserisci nuovo Anno di Pubblicazione ({book[3]}): ') or book[3])
            genere = input(f'Inserisci genere( FANTASY | HORROR | THRILLER | COMMEDIA | AZIONE) - ({book[4]}): ') or book[4]
        
            sql = """
                UPDATE libreria.books SET titolo=%s, autore=%s, 
                    anno_pubblicazione=%s, genere=%s WHERE id = %s;
            """
            values = (titolo, autore, anno_pubblicazione, genere, book[0])
            
            cursor.execute(sql, values) #Eseguo la query SQL passandogli i valori da inserire
            db.commit() # Completa l'operazione
            
            if cursor.rowcount > 0:
                print(f'Libro {titolo} di {autore} modificato nel DB')
            else:
                print('Nessun valore modificato')
            
        else:
            raise NameError('Nessun libro esistente')
    
    except NameError as e:
        print(e)
        return
    except ValueError as e:
        print('Anno in formato non valido.')
        return
    cursor.close()

def find_and_delete_book():
    cursor = db.cursor()
    
    try:
        print('Quale libro vuoi eliminare?')
        srcTitolo = input('Inserisci Titolo: ')
        srcAutore = input('Inserisci Autore: ')
            
        book = find_book(srcTitolo, srcAutore)
        if(book):
            sql = "DELETE FROM libreria.books WHERE id = %s"
            cursor.execute(sql, (book[0],)) #Eseguo la query SQL passandogli i valori da inserire
            db.commit() # Completa l'operazione
            if cursor.rowcount > 0:
                print(f'Libro {book[1]} di {book[2]} elimianto dal DB')
            else:
                print('Nessun valore eliminato')
        else:
            raise NameError('Nessun libro esistente')
    except NameError as e:
        print(e)
        return
    
    cursor.close()

def loan():
    # Implementare una funzione che consente di prestare un libro a un utente. 
    # Dovrà aggiornare lo stato del libro e registrare le informazioni del prestito.
    cursor = db.cursor()
    try:
        print('Quale libro vuoi prendere in prestito?')
        srcTitolo = input('Inserisci Titolo: ')
        srcAutore = input('Inserisci Autore: ')
        book = find_book(srcTitolo, srcAutore)
        if(book):
            if not book[5]:
                raise TypeError('Libro non disponibile')
            
            utente = input('Inserisic nome e cognome: ')
            
            sql = "INSERT INTO libreria.loans (utente, libro) VALUES (%s, %s)"
            values = (utente, book[0]) 
            cursor.execute(sql, values) #Eseguo la query SQL passandogli i valori da inserire
            
            sql = "UPDATE libreria.books SET disponibile = FALSE WHERE id = %s"
            cursor.execute(sql, (book[0],)) 
            
            db.commit() # Completa l'operazione
            
            print(f'Libro {book[1]} di {book[2]} preso in prestito da {utente}')
        else:
            raise NameError('Nessun libro esistente')
        
    except NameError as e:
        print(e)
    except TypeError as e:
        print(e)
    except Exception as e:
        print(e)
        db.rollback()
    
    cursor.close()
    
def back():
    # Restituzione libro: Implementare una funzione che consente di restituire un libro, 
    # aggiornando lo stato del libro e rimuovendo il prestito dalla lista.
    cursor = db.cursor()
    try:
        print('Quale libro vuoi restituire?')
        srcTitolo = input('Inserisci Titolo: ')
        srcAutore = input('Inserisci Autore: ')
        book = find_book(srcTitolo, srcAutore)
        
        if(book):
            if book[5]:
                raise TypeError('Libro già disponibile!')
            
            sql = """UPDATE libreria.loans SET data_restituzione = %s 
                    WHERE libro = %s AND data_restituzione IS NULL"""
                    
                    
            print(date.datetime.today(), book[0])
            values = (date.datetime.today(), book[0]) 
            cursor.execute(sql, values) #Eseguo la query SQL passandogli i valori da inserire
            
            sql = "UPDATE libreria.books SET disponibile = TRUE WHERE id = %s"
            cursor.execute(sql, (book[0],)) 
            
            db.commit() # Completa l'operazione
            
            print(f'Libro {book[1]} di {book[2]} restituito')
            
        else:
            raise NameError('Nessun libro esistente')
    except NameError as e:
        print(e)
    except Exception as e:
        print(e)
        db.rollback() 
    
    
    cursor.close()
    
 
# Il tradimento
# Elle Kennedy
    
# add_book()
# find_books()
# find_book_byID(3)
# find_and_update_book()
# find_and_delete_book()
# loan()   
# back()